# Tâche 4

In [1]:
import numpy as np
from mp_api.client import MPRester
from pymatgen.core.operations import SymmOp
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from jupyter_jsmol.pymatgen import quick_view
from lmapr1492 import plot_brillouin_zone, get_plot_bs, get_plot_dos, get_plot_bs_and_dos, get_branch_wavevectors
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from IPython.display import Image

In [2]:
mp_key = "xaEW7gxeGjtHSTeJuWSz9Uf8JpzNlgtg"
mp_id = "mp-9382"

In [3]:
with MPRester(mp_key) as m:
    structure = m.get_structure_by_material_id(mp_id)
struct = SpacegroupAnalyzer(structure)
prim_struct = struct.get_primitive_standard_structure()
conv_struct = struct.get_conventional_standard_structure()

Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

# Notions préalables

Un solide cristallin est un solide à structure régulière et périodique formé d'un empilement ordonné d'un grand nombre d'atomes, de molécules ou d'ions. Cette organisation repose sur un réseau ponctuel, c’est-à-dire un ensemble régulier de points dans l’espace représentant la périodicité du cristal. À chaque point de ce réseau est associé un groupement identique d’entités (atomes, ions ou molécules), toujours disposées de la même manière. L’ensemble ainsi formé constitue le réseau direct, qui décrit la structure réelle du cristal.

# Réseau direct

Le réseau direct est défini par ses vecteurs de base a, b et c, linéairement indépendants, tels que tout point du réseau peut s’écrire sous la forme :

$$
\mathbf{R} = l\mathbf{a} + m\mathbf{b} + n\mathbf{c}
$$

avec $ l, m, n \in \mathbb{Z} $.

Pour déterminer ces vecteurs, on se base sur la maille primitive. Celle-ci correspond à la plus petite unité répétitive du réseau, sans redondance. Elle garantit une description minimale et efficace de la structure cristalline, et facilite la construction du réseau réciproque.

Les vecteurs de base de la maille primitive (a, b, c) définissent les translations élémentaires du réseau cristallin. Ce sont les plus petits vecteurs qui permettent de reconstruire l’ensemble du réseau par répétition périodique dans les trois directions de l’espace.


In [4]:
# Extraction des vecteurs de base du réseau direct
res_direct = prim_struct.lattice
mat_direct = res_direct.matrix


lengths = [np.linalg.norm(vec) for vec in mat_direct]

print("Vecteurs de base du réseau direct :")
print(f"a = {mat_direct[0]}")
print(f"b = {mat_direct[1]}")
print(f"c = {mat_direct[2]}\n")

print("Longueurs des vecteurs :")
print(f"||a|| = {lengths[0]:.4f} Å")
print(f"||b|| = {lengths[1]:.4f} Å")
print(f"||c|| = {lengths[2]:.4f} Å")


Vecteurs de base du réseau direct :
a = [ 6.00163556 -1.68831424  0.        ]
b = [6.00163556 1.68831424 0.        ]
c = [5.52669753 0.         2.88542003]

Longueurs des vecteurs :
||a|| = 6.2346 Å
||b|| = 6.2346 Å
||c|| = 6.2346 Å


# Réseau réciproque

Le réseau réciproque d’un réseau de Bravais est défini comme l’ensemble des vecteurs $\mathbf{K} $ tels que :

$$
e^{i \mathbf{K} \cdot \mathbf{R}} = 1
$$

pour tout vecteur $\mathbf{R} $ appartenant au réseau direct. En diffraction, 𝐾 décrit la variation de vecteur d’onde permise par la structure périodique du cristal (k'-k). 

Cela implique que le produit scalaire $ \mathbf{K} \cdot \mathbf{R} $ est un multiple entier de $2\pi $, ce qui garantit une périodicité parfaite dans l’espace réciproque. Le réseau réciproque est lui-même un réseau de Bravais, et son réseau réciproque est le réseau de Bravais d’origine.

De manière analogue au réseau direct, nous utilisons la maille primitive pour construire les vecteurs de base du réseau réciproque. Ces vecteurs, notés $\mathbf{a}^*, \mathbf{b}^*, \mathbf{c}^* $, définissent la structure périodique du cristal dans l’espace réciproque.

Les expressions des vecteurs de base du réseau réciproque en fonction des vecteurs du réseau direct $ \mathbf{a}, \mathbf{b}, \mathbf{c} $ sont données par :

$$
\mathbf{a}^* = 2\pi \frac{\mathbf{b} \times \mathbf{c}}{\mathbf{a} \cdot (\mathbf{b} \times \mathbf{c})}, \quad 
\mathbf{b}^* = 2\pi \frac{\mathbf{c} \times \mathbf{a}}{\mathbf{a} \cdot (\mathbf{b} \times \mathbf{c})}, \quad 
\mathbf{c}^* = 2\pi \frac{\mathbf{a} \times \mathbf{b}}{\mathbf{a} \cdot (\mathbf{b} \times \mathbf{c})}
$$



In [5]:
# Extraction des vecteurs de base du réseau réciproque
res_reci = prim_struct.lattice.reciprocal_lattice
mat_reci = res_reci.matrix

lengths_reci = [np.linalg.norm(vec) for vec in mat_reci]

print("Vecteurs de base du réseau réciproque :")
print(f"a* = {mat_reci[0]}")
print(f"b* = {mat_reci[1]}")
print(f"c* = {mat_reci[2]}\n")

print("Longueurs des vecteurs réciproques :")
print(f"||a*|| = {lengths_reci[0]:.4f} Å⁻¹")
print(f"||b*|| = {lengths_reci[1]:.4f} Å⁻¹")
print(f"||c*|| = {lengths_reci[2]:.4f} Å⁻¹")

Vecteurs de base du réseau réciproque :
a* = [ 0.52345608 -1.86078669 -1.00262126]
b* = [ 0.52345608  1.86078669 -1.00262126]
c* = [0.         0.         2.17756349]

Longueurs des vecteurs réciproques :
||a*|| = 2.1776 Å⁻¹
||b*|| = 2.1776 Å⁻¹
||c*|| = 2.1776 Å⁻¹


# Sytème cristallin, type de maille et groupe ponctuel

Pour caractériser une structure cristalline, il est essentiel de déterminer son système cristallin, son type de maille et son groupe ponctuel. Le système cristallin décrit la manière dont les points du réseau se répètent périodiquement dans l’espace en fonction des paramètres de maille. Le type de maille définit l'unité de base du réseau, qui, par translation, permet de reconstruire l’ensemble de la structure cristalline. Enfin, le groupe ponctuel regroupe les opérations de symétrie qui laissent au moins un point invariant de l'espace sur lequel ces opérations de symétrie agissent.

In [6]:
syst_cristal = struct.get_crystal_system() 
print("Système cristallin :", syst_cristal)
print("")

type_maille = struct.get_lattice_type()
print("Type de maille :", type_maille)
print("")

group_ponctuel = struct.get_point_group_symbol()
print("Groupe ponctuel :", group_ponctuel)

Système cristallin : trigonal

Type de maille : rhombohedral

Groupe ponctuel : -3m


Le système cristallin est trigonal. Ce système se caractérise par une symétrie de rotation d'ordre 3, c’est-à-dire une invariance par rotation de $120^\circ$autour d’un axe principal.


La maille rhomboédrique est une maille primitive dont les trois vecteurs de base ont la même longueur et forment entre eux des angles égaux, mais différents de $90^\circ$

Le groupe ponctuel associé est le groupe $\overline{3}m $ Ce groupe indique la présence d’un axe de rotoinversion d’ordre 3 (une rotation d’ordre 3 suivie d’une inversion centrale), ainsi que de plans miroirs perpendiculaires à cet axe.
